# Space Launch Success Prediction (1957-2020)

This notebook looks at historical orbital rocket launch data and builds a simple model
to predict whether a launch will succeed or fail.

Data source: Kaggle dataset "All Space Missions from 1957"


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

%matplotlib inline


In [ ]:
import kagglehub

# Download the dataset (this saves it to a local cache folder)
path = kagglehub.dataset_download("agirlcoding/all-space-missions-from-1957")
print("Path to dataset files:", path)


In [ ]:
import os

# Check what files are inside the downloaded folder
print(os.listdir(path))


In [ ]:
# Load the csv file (change the file name below if it is different on your machine)
file_name = os.listdir(path)[0]
df = pd.read_csv(os.path.join(path, file_name))

df.head()


## 1. First look at the data

In [ ]:
print(df.shape)
print(df.columns)
df.info()


In [ ]:
# Check how many missing values are in each column
df.isnull().sum()


## 2. Cleaning the data

We need to:
- Convert the date column into a proper date format
- Pull the rocket family name out of the 'Detail' column
- Pull the country out of the 'Location' column
- Create a simple success / failure target column


In [ ]:
# Some rows may have bad dates, so errors='coerce' turns those into NaT (missing)
df['Datum'] = pd.to_datetime(df['Datum'], errors='coerce', utc=True)

df['Year'] = df['Datum'].dt.year
df['Decade'] = (df['Year'] // 10) * 10

df[['Datum', 'Year', 'Decade']].head()


In [ ]:
# The 'Detail' column looks like: "Falcon 9 Block 5 | Starlink V1 L9"
# We only want the rocket name part, before the '|'

def get_rocket_name(detail):
    if pd.isna(detail):
        return "Unknown"
    rocket_part = detail.split('|')[0].strip()
    # keep only the first two words, so different versions of the same rocket
    # get grouped together, e.g. "Falcon 9 Block 5" -> "Falcon 9"
    words = rocket_part.split()
    return " ".join(words[:2])

df['Rocket_Family'] = df['Detail'].apply(get_rocket_name)

df['Rocket_Family'].value_counts().head(10)


In [ ]:
# The 'Location' column looks like: "Site 31/6, Baikonur Cosmodrome, Kazakhstan"
# The country is usually the last part after the last comma

df['Country'] = df['Location'].apply(lambda x: str(x).split(',')[-1].strip())

df['Country'].value_counts().head(10)


In [ ]:
# Status Mission usually has values like: Success, Failure, Partial Failure, Prelaunch Failure
print(df['Status Mission'].value_counts())

# Simple target: 1 if mission was a success, 0 for anything else
df['Target'] = (df['Status Mission'] == 'Success').astype(int)

df['Target'].value_counts()


## 3. Exploring the data (EDA)

In [ ]:
success_by_decade = df.groupby('Decade')['Target'].mean()

plt.figure(figsize=(8,5))
success_by_decade.plot(kind='bar', color='skyblue')
plt.title('Launch success rate by decade')
plt.ylabel('Success rate')
plt.xlabel('Decade')
plt.ylim(0,1)
plt.show()


In [ ]:
top_companies = df['Company Name'].value_counts().head(10)
print(top_companies)

success_by_company = (
    df[df['Company Name'].isin(top_companies.index)]
    .groupby('Company Name')['Target'].mean()
    .sort_values()
)

plt.figure(figsize=(8,5))
success_by_company.plot(kind='barh', color='lightgreen')
plt.title('Success rate for top 10 companies (by number of launches)')
plt.xlabel('Success rate')
plt.show()


## 4. Feature engineering

The most important feature for predicting success is usually how many times a rocket
has already flown. A rocket on its 1st flight is much riskier than one on its 100th flight.


In [ ]:
# Sort by date so launches are counted in the correct order
df = df.sort_values('Datum')

# For each rocket family, count which launch number this is (1st, 2nd, 3rd ...)
df['Flight_Number'] = df.groupby('Rocket_Family').cumcount() + 1

df[['Rocket_Family', 'Datum', 'Flight_Number']].head(10)


In [ ]:
# Group flight numbers into bins so the trend is easier to see
df['Flight_Number_Bin'] = pd.cut(
    df['Flight_Number'],
    bins=[0, 5, 10, 20, 50, 100, 1000],
    labels=['1-5', '6-10', '11-20', '21-50', '51-100', '100+']
)

success_by_experience = df.groupby('Flight_Number_Bin')['Target'].mean()

plt.figure(figsize=(8,5))
success_by_experience.plot(kind='bar', color='salmon')
plt.title('Success rate vs how many times the rocket family has flown before')
plt.ylabel('Success rate')
plt.xlabel('Flight number of rocket family')
plt.ylim(0,1)
plt.show()


## 5. Preparing data for the model

In [ ]:
# Drop rows where the date could not be read
model_df = df.dropna(subset=['Year']).copy()

# Keep only the top 15 rocket families and top 15 companies.
# Everything else becomes "Other" so we don't end up with hundreds of tiny categories.
top_rockets = model_df['Rocket_Family'].value_counts().head(15).index
top_companies_list = model_df['Company Name'].value_counts().head(15).index

model_df['Rocket_Family_grouped'] = model_df['Rocket_Family'].apply(
    lambda x: x if x in top_rockets else 'Other'
)
model_df['Company_grouped'] = model_df['Company Name'].apply(
    lambda x: x if x in top_companies_list else 'Other'
)

# One-hot encode the categorical columns
features = pd.get_dummies(
    model_df[['Rocket_Family_grouped', 'Company_grouped', 'Country']],
    drop_first=True
)

# Add the numeric columns
features['Year'] = model_df['Year']
features['Flight_Number'] = model_df['Flight_Number']

target = model_df['Target']

print(features.shape)
features.head()


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, random_state=42, stratify=target
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])
print("Success rate in training data:", round(y_train.mean(), 3))


## 6. Logistic Regression model

Logistic Regression is a simple model that works well as a first baseline. Since most
launches are successful, we use class_weight='balanced' so the model does not just
learn to always predict "success".


In [ ]:
log_model = LogisticRegression(max_iter=1000, class_weight='balanced')
log_model.fit(X_train, y_train)

log_preds = log_model.predict(X_test)
log_probs = log_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, log_preds))
print("ROC AUC score:", round(roc_auc_score(y_test, log_probs), 3))


## 7. Random Forest model

Random Forest usually performs better than Logistic Regression on this kind of data,
and it also tells us which features were most useful.


In [ ]:
rf_model = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_test)
rf_probs = rf_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, rf_preds))
print("ROC AUC score:", round(roc_auc_score(y_test, rf_probs), 3))


In [ ]:
cm = confusion_matrix(y_test, rf_preds)

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion matrix - Random Forest')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


In [ ]:
importance = pd.Series(rf_model.feature_importances_, index=features.columns)
top_features = importance.sort_values(ascending=False).head(10)

plt.figure(figsize=(8,5))
top_features.plot(kind='barh')
plt.title('Top 10 most important features (Random Forest)')
plt.xlabel('Importance')
plt.gca().invert_yaxis()
plt.show()


## 8. Reliability growth - Falcon 9 example

Here we look at one rocket family (Falcon 9) and check how its failure rate changed
as it flew more missions. This is a simple version of what is called a "reliability
growth curve" in aerospace engineering.


In [ ]:
falcon9 = df[df['Rocket_Family'] == 'Falcon 9'].copy()
print("Total Falcon 9 launches in data:", falcon9.shape[0])

# Group flight numbers into bins of 25 launches
falcon9['Flight_Bin'] = pd.cut(
    falcon9['Flight_Number'],
    bins=range(0, int(falcon9['Flight_Number'].max()) + 25, 25)
)

failure_rate_by_bin = 1 - falcon9.groupby('Flight_Bin')['Target'].mean()

plt.figure(figsize=(8,5))
failure_rate_by_bin.plot(kind='line', marker='o', color='red')
plt.title('Falcon 9 failure rate as it gains more flights')
plt.ylabel('Failure rate')
plt.xlabel('Flight number range')
plt.xticks(rotation=45)
plt.show()


## 9. Saving output files

These files can be used later in Excel, Power BI, or Tableau.


In [ ]:
# Save the cleaned dataset
df.to_csv('cleaned_space_missions.csv', index=False)

# Save a summary table - launches and success rate per company
company_summary = df.groupby('Company Name').agg(
    Total_Launches=('Target', 'count'),
    Success_Rate=('Target', 'mean')
).sort_values('Total_Launches', ascending=False)

company_summary.to_csv('company_summary.csv')

company_summary.head(10)


## 10. Notes / next steps

- Flight number (how many times a rocket family has flown before) was expected to be
  one of the strongest predictors - check the feature importance chart above to confirm.
- A more advanced version of this project could use an LSTM or a proper Bayesian
  reliability growth model (Duane model) instead of simple binning.
- Some rocket family names may not be grouped perfectly because of how the 'Detail'
  column is written. This can be cleaned further by hand if needed, especially for
  rockets with unusual naming patterns.
- Column names in the actual downloaded file might be slightly different
  (e.g. 'Status Mission' vs 'Status_Mission') - run df.columns first and adjust
  if needed.
